In [ ]:
# Instala o Google Chrome oficial para rodar no Colab
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb

# Instala o webdriver-manager para gerenciar o driver automaticamente
!pip install selenium webdriver-manager

In [ ]:
import glob
import re
import time
import random
import pandas as pd
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
def criar_driver():
  options = Options()
  options.add_argument('--headless')
  options.add_argument("--start-maximized")
  options.add_argument('--no-sandbox')
  options.add_argument('--disable-dev-shm-usage')
  options.add_argument('--remote-debugging-port=9222')
  options.add_argument("--disable-gpu")

  service = Service(ChromeDriverManager().install())

  return webdriver.Chrome(service=service, options=options)

In [ ]:
df_origem = pd.read_csv("PROCESSOS_AMBIENTAIS_TRF1_1985_2025.csv", dtype=str)
processos = df_origem['numero'].str.replace("'", "").str.strip().tolist()

In [ ]:
COLUNAS = ["Número do Processo", "Data de Distribuição", "Classe Judicial", "Assunto",
           "Jurisdição", "Órgão Julgador", "Status", "Polo Ativo", "Polo Passivo", "Outros Interessados"]

In [ ]:
def extrair_participantes(driver, xpath_table):
    participantes = []
    rows = driver.find_elements(By.XPATH, xpath_table)
    for row in rows:
        try:
            name = row.find_element(By.XPATH, ".//td[1]//span[contains(@class,'text-bold') or contains(@class,'')]").text.strip()
            status_p = row.find_element(By.XPATH, ".//td[2]//div").text.strip()
            participantes.append(f"{name} [{status_p}]")
        except:
            continue
    return ", ".join(participantes) if participantes else ""

In [ ]:
def buscar_processo(driver, wait, processo_num):
    input_pesquisa = wait.until(EC.visibility_of_element_located(
        (By.ID, "fPP:numProcesso-inputNumeroProcessoDecoration:numProcesso-inputNumeroProcesso")
    ))

    driver.execute_script("arguments[0].value = '';", input_pesquisa)
    input_pesquisa.click()
    time.sleep(0.5)

    for digito in processo_num:
        input_pesquisa.send_keys(digito)

    time.sleep(1)

    btn_pesquisa = wait.until(EC.element_to_be_clickable((By.ID, "fPP:searchProcessos")))
    btn_pesquisa.click()
    time.sleep(2)

In [ ]:
def abrir_detalhes(driver, wait):
    try:
        link = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//a[@title='Ver Detalhes']")
        ))
        driver.execute_script("arguments[0].click();", link)
        return True
    except:
        return False

In [ ]:
def trocar_para_nova_aba(driver, wait):
    original = driver.current_window_handle
    wait.until(EC.number_of_windows_to_be(2))

    for handle in driver.window_handles:
        if handle != original:
            driver.switch_to.window(handle)
            return original

In [ ]:
def extrair_dados_processo(driver):
    res = {}

    res["Data de Distribuição"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Data da Distribuição')]/../../div[2]"
    ).text.strip()

    res["Classe Judicial"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Classe Judicial')]/../../div[2]"
    ).text.strip()

    res["Assunto"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Assunto')]/../../div[2]/div"
    ).text.strip()

    res["Jurisdição"] = driver.find_element(
        By.XPATH, "//label[contains(text(), 'Jurisdição')]/../../div[2]"
    ).text.strip()

    res["Órgão Julgador"] = driver.find_element(
        By.XPATH, "//div[contains(@class, 'value') and .//b[text()='Órgão Julgador']]/div"
    ).text.replace("Órgão Julgador", "").strip()

    tb_status = driver.find_elements(
        By.XPATH, "//tbody[contains(@id, 'processoEvento')]/tr/td//span"
    )

    res["Status"] = tb_status[0].text.strip() if tb_status else "Sem movimentação"

    res["Polo Ativo"] = extrair_participantes(driver, "//tbody[contains(@id, 'processoPartesPoloAtivoResumidoTableBinding')]/tr")
    res["Polo Passivo"] = extrair_participantes(driver, "//tbody[contains(@id, 'processoPartesPoloPassivoResumidoTableBinding')]/tr")
    res["Outros Interessados"] = extrair_participantes(driver, "//tbody[contains(@id, 'processoParteOutrosInteressadosResumidoTableBinding')]/tr")

    return res

In [ ]:
def verificar_nao_encontrado(driver):
    msg = driver.find_elements(By.CSS_SELECTOR, "span.rich-messages-label")
    if msg and "não encontrou nenhum processo" in msg[0].text.lower():
        return True
    return False

In [ ]:
def processar_processo(driver, wait, processo_num):
    res = {col: "" for col in COLUNAS}
    res["Número do Processo"] = processo_num

    try:
        buscar_processo(driver, wait, processo_num)

        if not abrir_detalhes(driver, wait):
            if verificar_nao_encontrado(driver):
                res["Status"] = "Não encontrado"
            else:
                res["Status"] = "Erro no carregamento"
            return res

        original = trocar_para_nova_aba(driver, wait)

        dados = extrair_dados_processo(driver)
        res.update(dados)

        driver.close()
        driver.switch_to.window(original)

    except Exception as e:
        res["Status"] = f"Erro: {type(e).__name__}"

        if len(driver.window_handles) > 1:
            try:
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            except:
                pass

    return res

In [ ]:
def processar_batch(lote_inicial=1):
    batch_size = 20 # poucos por lote para evitar bloqueio
    total_processos = len(processos)

    for i in range((lote_inicial - 1) * batch_size, total_processos, batch_size):
        batch = processos[i:i+batch_size]
        batch_num = i // batch_size + 1
        total_batches = (total_processos + batch_size - 1) // batch_size

        print(f"\n--- Lote {batch_num} de {total_batches} ---")

        resultados = []

        driver = criar_driver()
        wait = WebDriverWait(driver, 30)

        try:
            driver.get("https://pje1g.trf1.jus.br/consultapublica/ConsultaPublica/listView.seam")

            for processo_num in batch:
                print(f"Buscando processo {processo_num}")

                res = processar_processo(driver, wait, processo_num)
                resultados.append(res)

                print(f"✅ {processo_num} -> {res.get('Status', '')}")

                # delay humano
                time.sleep(random.uniform(3, 6))

        except Exception as e:
            print(f"❌ Erro no lote {batch_num}: {e}")

        finally:
            if resultados:
                df_batch = pd.DataFrame(resultados)
                nome_saida = f"resultados_lote_{batch_num}.xlsx"
                df_batch.to_excel(nome_saida, index=False)
                print(f"Lote {batch_num} salvo em {nome_saida}")

            try:
                driver.quit()
            except:
                pass

            # pausa humana entre lotes
            sleep_time = random.uniform(30, 90)
            print(f"Pausa de {sleep_time:.1f}s...")
            time.sleep(sleep_time)

In [ ]:
processar_batch() # incluir de qual lote deseja começar, caso não seja desde o começo

In [ ]:
# Unifica os arquivos xlsx salvos
arquivos = glob.glob("resultados_lote_*.xlsx")

df_final = pd.concat([pd.read_excel(f) for f in arquivos], ignore_index=True)
df_final.to_excel("trf1_completos.xlsx", index=False)

print("Todos os batches foram reunidos em trf1_completos.xlsx")